# SOXX constituent basket lead information screen

This notebook is the primary research entrypoint for the statically reviewed plan in `plan.json`. The screen compares fixed OLS model B (SOXX first-hour return) with model C (the same input plus the prior-public 30-member basket return less SOXX). It uses native Qlib calendars, bounded `D.features` reads, phase-specific `DatasetH` objects, and a Qlib Recorder.

**Execution requires the separately sealed owner GO.** The 2024 and 2025 market-data reads and fits require the research owner's separate GO after checking the final plan, exact source hashes, implementation identity, and normalized notebook snapshot. No pre-existing Recorder is read. The 2024 models and receipt must be sealed before any 2025 price, label, or weight construction.

The planned target is `log(SOXX open at right-edge 11:01 / open at right-edge 10:33)`, corresponding to 11:00 and 10:32 fills. The 10:30 factor uses completed 12-bar 5-minute constituent prefixes and 60-bar 1-minute SOXX prefixes. A session stays on the native calendar when data are unknown; incomplete baskets are never partially renormalized.

The sole primary gate is positive fitted C gamma, positive 2025 paired daily squared-loss improvement, and a positive fixed one-sided stationary-bootstrap lower bound (63-session mean block, 10,000 replicates, seed 20261014, quantile 0.025/15). A result is a reused-history information screen only.

In [1]:
from pathlib import Path
import sys

ROOT = Path('/Users/rziiiii/dev/workspace')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.workflow.basket_etf_lead_information import run

# This call rejects missing/stale GO before market IO, fits or Recorder creation.
SUMMARY = run(root=ROOT)
SUMMARY


## Frozen execution sequence

1. Write the attempt journal before source reads, then validate exact plan, design review, implementation and pinned file bytes.
2. Read only 2024 Qlib source calendars and the two bounded price matrices; build 2024 PIT weights from SEC filing metadata and a phase-only native calendar. Preserve unknown calendar rows.
3. Build native Qlib DatasetH inputs, require at least 200 complete rows, fit B and C once, and save source identities, scaler means/stds, coefficients, masks, and hashes in the new Recorder. Finish and read back the sealed freeze receipt.
4. Only after that readback, query 2025 prices and build 2025 PIT weights. Score the unchanged coefficients, write paired losses and the fixed bootstrap to the same Recorder, and close it.
5. Recheck pinned bytes and source identities. Leave unknown rows explicit and report any contract failure without retrying or silently changing the design.

Reusable mechanics live in `src/workflow/basket_etf_lead_information.py`. This static handoff has no numerical result, Recorder ID, or promotion status.

# 成分篮子领先 SOXX：固定信息检验结论

结果：**not_promoted**，该固定候选关闭，不进入费用账户。总目标仍是找到费用后稳定跑赢 SOXX 的有效因子。

使用2024年252个完整日拟合B/C两个固定OLS；模型与训练期标准化封存读回后，才读取2025年250个完整日。B仅含SOXX同期第一小时收益，C增加30成员完整披露篮子相对SOXX的价格变化。篮子先合成加权简单收益再取对数，避免几何收益带来的机械离散度残差；不是实时iNAV或可套利报价。10:30闭合信息假定10:31可用，主要标签是10:32开盘到11:00开盘的SOXX收益。

|冻结门槛/指标|结果|
|---|---|
|2024拟合方向 γ>0|0.597877，通过|
|2025基线B均方误差|2.15966287138e-05|
|2025候选C均方误差|2.21919092132e-05|
|C相对B误差变化|增加2.76%|
|配对误差改善 MSE_B−MSE_C|-5.95280499344e-07，失败|
|固定区块置信下界|-2.00519100536e-06，失败|
|增量预测与基线残差相关|-0.023668，仅诊断|

两年全部502个日历行都合格，未知特征/标签为0。2024年完整权重中保留229个WOLF成员日；2025年按实际已披露篮子不再包含WOLF，没有以缺价子集重新归一化。旧PIT二进制库缺WOLF的问题通过已校验SEC源构建分期原生DatasetH解决；没有改旧库或增加外部行情源。

Qlib Recorder `c0c07a824ad8495daa5a19db4d793a8a`，2次信息拟合、1次固定63日均块/10000次/seed20261014/alpha=.025/15统计检验、0个金融账户。模型、原生Dataset、全部预测/误差和已有bootstrap抽样保存在Recorder，NAS归档SHA `539842e6db512afdb2e4ffed72a297bdaaf2d8525e1522b59c2a1dc54f756001`。

冷恢复：1次NAS下载、两次各11对象读回；首次仅为核验脚本把source_code的sha256/use字典当裸哈希，复用同一恢复副本完成核对。21项artifact、源码、冻结模型、Dataset标签和保存的bootstrap分位数一致，没有重训/回放/重抽统计样本。

限制：2024–2025已被项目其他研究使用，不是独立前向证据；固定60秒可用时点未经历史首次接收认证，披露权重存在滞后，成交bar不是ETF/成分可成交双边报价。研究动机来自 [Box等（2021）](https://www.andrewalynch.com/papers/2021-etf-arbitrage.pdf)；文献中的快速价格发现不能据此推导本项目存在可交易超额。

下一步：关闭这个固定篮子领先版本，不调整符号、切点、标签窗口或筛选年份。另行核查预拟定的“下行波动占比对未来风险的增量信息”是否与旧H5和无符号风险路线有实质区别；任何通过都仍需完整费用后SOXX比较。
